In [ ]:
# Instalaivimas reikalingų paketų
#!pip install git+https://github.com/huggingface/transformers accelerate
print("Instaliavimas baigtas")

In [ ]:
# Perspėjimų ignoravimas
import warnings
warnings.filterwarnings("ignore")  # Ignore all warnings

from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import requests
import torch
import os
import re
import pandas as pd
print("Importavimas baigtas")

In [ ]:
# Modelių naudojimas
#model_id = "Ertugrul/Qwen2.5-VL-7B-Captioner-Relaxed"
model_id = "Qwen/Qwen2-VL-7B-Instruct"

model = AutoModelForImageTextToText.from_pretrained(
  model_id,
  device_map="auto",
  torch_dtype=torch.bfloat16,
  attn_implementation="flash_attention_2",
)

min_pixels = 256*28*28
max_pixels = 1280*28*28

processor = AutoProcessor.from_pretrained(model_id, max_pixels=max_pixels, min_pixels=min_pixels)

print("Modelio krovimas baigtas")

In [ ]:
# Lentelės užkrovimas, generavimas ir saugojimas
df = pd.read_csv('flicker8k_edited.csv', sep=';')
import numpy as np

# Išsivalome lentelę
columns_to_clear = [
    'BLEU', 'ROGUE', 'METEOR',
    'BERTscore Precision', 'BERTscore Recall', 'BERTscore F1', 'SentenceBERT'
]
for col in columns_to_clear:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: '')

image_folder = "Flickr8k"

#for index, row in df.head(20).iterrows():
for index, row in df.iterrows():
    filename = row["Image Name"]
    image_path = os.path.join(image_folder, filename)
    try:
        system_message = "You are an expert image describer."
        def generate_description(path, model, processor):
            image = Image.open(path).convert("RGB")
            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": system_message}],
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "Keep it short. No more than 10 words. The text is the caption for the image. Do not write any additional text."},
                        {"type": "image", "image": image},
                    ],
                },
            ]
            text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
            )

            inputs = processor(
            text=[text],
            images=image,
            padding=True,
            return_tensors="pt",
            )
            inputs = inputs.to(model.device)
    
            generated_ids = model.generate(**inputs, max_new_tokens=50, min_p=0.1, do_sample=True, temperature=1.5)
            generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
            output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
            return output_text[0]

        generated_caption = generate_description(image_path, model, processor)
        # Įrašymas į failą
        df.at[index, "Generated Caption"] = generated_caption

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        df.at[index, "Generated Caption"] = f"Error: {e}"

# Saugojimas
df.to_csv("Results/flicker8k_en_16.csv", sep=';', index=False)
print("Generavimas baigtas.")